In [1]:
import sys
!{sys.executable} -m pip install pyodbc
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import pytz
import yfinance as yf
import pyodbc
!{sys.executable} -m pip install matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import logging
import pandas as pd
from truedata import TD_hist
import requests




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
def fetch_truedata_history(
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    username = 'tdwsf695'
    password = 'ocean@695'
    # Initialize connection
    td_hist = TD_hist(username, password)
    df_list = []
    error_list = []
    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            # Check column names and rename accordingly
            rename_dict = {}
            if 'timestamp' in df.columns:
                rename_dict['timestamp'] = 'Date'
            elif 'datetime' in df.columns:
                rename_dict['datetime'] = 'Date'
            elif 'date' in df.columns:
                rename_dict['date'] = 'Date'
            rename_dict.update({
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })
            df = df.rename(columns=rename_dict)

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list



In [3]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

def process_portfolio(nav_df, ticker_data, initial_value=75, inception_date=None, output_file=None):
    """
    Process portfolio allocation with month-by-month rebalancing.

    Parameters
    ----------
    nav_df : pd.DataFrame
        Dataframe with at least ['Date', 'Year-Month', 'Ticker'] columns.
    ticker_data : pd.DataFrame
        Historical OHLC data with ['Date', 'Ticker', 'Close'].
    initial_value : float
        Initial portfolio allocation value.
    inception_date : str or pd.Timestamp or None
        Start date from which holdings should be valued.
    output_file : str or None
        If provided, saves the final dataframe to Excel.

    Returns
    -------
    pd.DataFrame
        Final dataframe with portfolio values.
    """
    df_lis = []
    last_month_value = {}
    last_month_quantity = {}

    nav_df = nav_df.sort_values(['Date', 'Ticker']).copy()
    nav_df['Date'] = pd.to_datetime(nav_df['Date'])
    ticker_data = ticker_data.sort_values(['Ticker', 'Date']).copy()
    ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])

    if inception_date is None:
        inception_date = nav_df['Date'].min()
    else:
        inception_date = pd.to_datetime(inception_date)

    for year_month in nav_df['Year-Month'].drop_duplicates():
        month_nav = nav_df[nav_df['Year-Month'] == year_month].copy()
        tickers = month_nav['Ticker'].dropna().unique().tolist()
        selection_date = pd.to_datetime(month_nav['Date'].min())
        year_month_date = pd.to_datetime(f"{year_month}-01")

        prev_month_start = year_month_date - relativedelta(months=2)
        curr_month_start = year_month_date
        curr_month_end = year_month_date + pd.offsets.MonthEnd(0)

        stock_data = ticker_data[
            (ticker_data['Date'] >= prev_month_start)
            & (ticker_data['Date'] <= curr_month_end)
            & (ticker_data['Ticker'].isin(tickers))
        ].copy()
        stock_data = stock_data[stock_data['Date'] >= inception_date].copy()
        stock_data['%change'] = stock_data.groupby('Ticker')['Close'].pct_change()

        stock_data_flt = stock_data[
            (stock_data['Date'] >= curr_month_start) & (stock_data['Date'] <= curr_month_end)
        ].copy()
        if stock_data_flt.empty:
            continue

        if not last_month_value:
            allocation_per_stock = initial_value / len(tickers)
            stock_allocations = {ticker: allocation_per_stock for ticker in tickers}
        else:
            stock_allocations = {ticker: last_month_value[ticker] for ticker in tickers if ticker in last_month_value}
            dropped_stocks = [ticker for ticker in last_month_value if ticker not in tickers]
            dropped_value = sum(last_month_value[ticker] for ticker in dropped_stocks)
            new_stocks = [ticker for ticker in tickers if ticker not in last_month_value]
            if new_stocks:
                allocation_per_stock = dropped_value / len(new_stocks) if dropped_value else 0.0
                for ticker in new_stocks:
                    stock_allocations[ticker] = allocation_per_stock

        for ticker, init_value in stock_allocations.items():
            ticker_index = stock_data_flt[stock_data_flt['Ticker'] == ticker].index
            ticker_df = stock_data_flt.loc[ticker_index].copy()
            if ticker_df.empty:
                continue

            stock_data_flt.loc[ticker_index, 'Initial_Allocation'] = init_value
            stock_data_flt.loc[ticker_index, 'Selection_Date'] = selection_date
            stock_data_flt.loc[ticker_index, 'Buy_Hold_Value'] = init_value * (
                (1 + stock_data_flt.loc[ticker_index, '%change'].fillna(0)).cumprod()
            )

            buy_price = float(ticker_df.iloc[0]['Close'])
            quantity = last_month_quantity.get(ticker, init_value / buy_price if buy_price else 0.0)
            stock_data_flt.loc[ticker_index, 'Buy_Price'] = buy_price
            stock_data_flt.loc[ticker_index, 'Quantity'] = quantity
            if 'Real_Rank' in month_nav.columns:
                stock_data_flt.loc[ticker_index, 'Real_Rank'] = month_nav.loc[month_nav['Ticker'] == ticker, 'Real_Rank'].iloc[0]

        last_month_quantity = stock_data_flt.groupby('Ticker')['Quantity'].last().to_dict()
        last_month_value = stock_data_flt.groupby('Ticker')['Buy_Hold_Value'].last().to_dict()
        stock_data_flt['Total_Portfolio_Value'] = stock_data_flt.groupby('Date')['Buy_Hold_Value'].transform('sum')
        df_lis.append(stock_data_flt)

    final_df = pd.concat(df_lis, ignore_index=True).sort_values(['Date', 'Ticker']).reset_index(drop=True)

    if output_file:
        final_df.to_excel(output_file, index=False)

    return final_df


def build_weighted_hedge_segment(hedge_prices, start_date, end_date, base_values, segment_name):
    segment = hedge_prices[
        (hedge_prices['Date'] >= start_date)
        & (hedge_prices['Date'] <= end_date)
        & (hedge_prices['Ticker'].isin(base_values))
    ][['Date', 'Ticker', 'Open', 'Close']].copy()
    if segment.empty:
        return segment

    segment = segment.sort_values(['Ticker', 'Date'])
    segment['%change'] = segment.groupby('Ticker')['Close'].pct_change()
    segment['Initial_Allocation'] = segment['Ticker'].map(base_values)
    segment['ret_factor'] = 1 + segment['%change'].fillna(0)
    segment['cum_factor'] = segment.groupby('Ticker')['ret_factor'].cumprod()
    segment['Buy_Hold_Value'] = segment['Initial_Allocation'] * segment['cum_factor']

    buy_prices = segment.groupby('Ticker')['Close'].transform('first')
    segment['Buy_Price'] = buy_prices
    segment['Quantity'] = np.where(buy_prices > 0, segment['Initial_Allocation'] / buy_prices, 0.0)
    segment['Selection_Date'] = start_date
    segment['Hedge_Segment'] = segment_name
    segment['Total_Portfolio_Value'] = segment.groupby('Date')['Buy_Hold_Value'].transform('sum')
    return segment.drop(columns=['ret_factor', 'cum_factor'])


def build_rebalanced_hedge_book(base_portfolio_df, portfolio_end_date=None, default_hedge_value=25.0):
    if portfolio_end_date is None:
        portfolio_end_date = pd.to_datetime(base_portfolio_df['Date']).max()
    else:
        portfolio_end_date = pd.to_datetime(portfolio_end_date)

    cutoff_date = pd.Timestamp('2025-11-30')
    if portfolio_end_date <= cutoff_date:
        return pd.DataFrame()

    hedge_start_factor = base_portfolio_df[
        (base_portfolio_df['Ticker'] == 'GOLDBEES') & (base_portfolio_df['Date'] <= cutoff_date)
    ].sort_values('Date')
    hedge_seed = float(hedge_start_factor['Buy_Hold_Value'].iloc[-1]) if not hedge_start_factor.empty else default_hedge_value

    hedge_prices = fetch_truedata_history(
        ticker_list=['GOLDBEES', 'SILVERBEES', 'MOGSEC', 'LIQUIDCASE'],
        duration='5 Y',
        bar_size='EOD',
        sleep_time=0.1
    )[0]

    segments = []

    decjan_start = pd.Timestamp('2025-12-01')
    decjan_end = min(pd.Timestamp('2026-01-31'), portfolio_end_date)
    if portfolio_end_date >= decjan_start:
        decjan_values = {'GOLDBEES': 0.60 * hedge_seed, 'SILVERBEES': 0.20 * hedge_seed, 'MOGSEC': 0.20 * hedge_seed}
        df_decjan = build_weighted_hedge_segment(
            hedge_prices,
            start_date=decjan_start,
            end_date=decjan_end,
            base_values=decjan_values,
            segment_name='2025-12_to_2026-01'
        )
        if not df_decjan.empty:
            segments.append(df_decjan)
        else:
            df_decjan = pd.DataFrame()
    else:
        df_decjan = pd.DataFrame()

    feb_start = pd.Timestamp('2026-02-01')
    feb_end = min(pd.Timestamp('2026-02-28'), portfolio_end_date)
    if portfolio_end_date >= feb_start and not df_decjan.empty:
        feb_factor = (
            df_decjan.groupby('Date', as_index=False)['Buy_Hold_Value'].sum().sort_values('Date')['Buy_Hold_Value'].iloc[-1]
        )
        feb_values = {'GOLDBEES': 0.40 * feb_factor, 'MOGSEC': 0.60 * feb_factor}
        df_feb = build_weighted_hedge_segment(
            hedge_prices,
            start_date=feb_start,
            end_date=feb_end,
            base_values=feb_values,
            segment_name='2026-02'
        )
        if not df_feb.empty:
            segments.append(df_feb)
        else:
            df_feb = pd.DataFrame()
    else:
        df_feb = pd.DataFrame()

    mar_start = pd.Timestamp('2026-03-01')
    if portfolio_end_date >= mar_start and not df_feb.empty:
        feb_last_date = df_feb['Date'].max()
        feb_last = df_feb[df_feb['Date'] == feb_last_date].set_index('Ticker')['Buy_Hold_Value'].to_dict()
        mar_values = {'GOLDBEES': feb_last.get('GOLDBEES', 0.0), 'LIQUIDCASE': feb_last.get('MOGSEC', 0.0)}
        df_mar = build_weighted_hedge_segment(
            hedge_prices,
            start_date=mar_start,
            end_date=portfolio_end_date,
            base_values=mar_values,
            segment_name='2026-03_onward'
        )
        if not df_mar.empty:
            segments.append(df_mar)

    if not segments:
        return pd.DataFrame()

    return pd.concat(segments, ignore_index=True).sort_values(['Date', 'Ticker']).reset_index(drop=True)

In [4]:
import os
import pandas as pd

def prepare_and_process_portfolio(input_file, start_date, end_date, output_folder,
                                  process_portfolio,
                                  equity_allocation=75, gold_allocation=25):
    """
    Prepare portfolio dataframe with momentum stocks + GOLDBEES and process performance.
    """

    nav_df_raw = pd.read_excel(input_file).rename(columns={'End_Date': 'Date'})
    nav_df_raw['Date'] = pd.to_datetime(nav_df_raw['Date'])

    selected_cols = ['Date', 'Ticker']
    if 'Real_Rank' in nav_df_raw.columns:
        selected_cols.append('Real_Rank')

    nav_df = (
        nav_df_raw[(nav_df_raw['Date'] >= start_date) & (nav_df_raw['Date'] <= end_date)]
        .reset_index(drop=True)[selected_cols]
    )
    nav_df['Year-Month'] = nav_df['Date'].dt.to_period('M').astype(str)

    goldbees_df = pd.DataFrame({
        'Date': nav_df['Date'].drop_duplicates().sort_values(),
        'Ticker': 'GOLDBEES'
    })
    if 'Real_Rank' in nav_df.columns:
        goldbees_df['Real_Rank'] = np.nan
    goldbees_df['Year-Month'] = pd.to_datetime(goldbees_df['Date']).dt.to_period('M').astype(str)

    concat_df = (
        pd.concat([nav_df, goldbees_df], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    ticker_df = concat_df.query("Ticker != 'GOLDBEES'")
    symbol_list = ticker_df['Ticker'].unique()
    ticker_data_other_stocks = fetch_truedata_history(
        ticker_list=symbol_list,
        duration='10 Y',
        bar_size='EOD',
        sleep_time=0.1
    )[0]

    gold_df = concat_df.query("Ticker == 'GOLDBEES'")
    symbol_list = gold_df['Ticker'].unique()
    ticker_data_gold = fetch_truedata_history(
        ticker_list=symbol_list,
        duration='10 Y',
        bar_size='EOD',
        sleep_time=0.1
    )[0]

    inception_date = pd.to_datetime(start_date)
    final_df_other_stocks = process_portfolio(
        ticker_df,
        ticker_data_other_stocks,
        equity_allocation,
        inception_date=inception_date
    )
    final_df_gold = process_portfolio(
        gold_df,
        ticker_data_gold,
        gold_allocation,
        inception_date=inception_date
    )

    final_df = (
        pd.concat([final_df_other_stocks, final_df_gold], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    middle_folder = os.path.basename(os.path.dirname(input_file))
    output_file = os.path.join(output_folder, f"{middle_folder}_gold_buy&hold_returns.xlsx")
    print(f"Final output path: {output_file}")

    return final_df

In [5]:
final_df = prepare_and_process_portfolio(
    input_file=r"C:\Users\LENOVO\Desktop\Ocean Finvest\MOMENTUM_DB_2 copy\Stocks\Nifty_500_2025_Apr_20_stocks_results\master_momentum_summary.xlsx",
    start_date="2023-04-01",
    end_date=date.today().strftime('%Y-%m-%d'),
    output_folder="Trials",
    process_portfolio=process_portfolio
)

final_df

(2026-04-29 09:49:36,638) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
2026-04-29 09:49:36,638 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-04-29 09:49:37,259 - INFO - Fetched data for ABCAPITAL (2145 rows).
2026-04-29 09:49:37,740 - INFO - Fetched data for ANANDRATHI (1085 rows).
2026-04-29 09:49:38,318 - INFO - Fetched data for ANANTRAJ (2478 rows).
2026-04-29 09:49:38,894 - INFO - Fetched data for BANKBARODA (2478 rows).
2026-04-29 09:49:39,542 - INFO - Fetched data for BANKINDIA (2478 rows).
2026-04-29 09:49:40,126 - INFO - Fetched data for CANBK (2478 rows).
2026-04-29 09:49:40,678 - INFO - Fetched data for CUMMINSIND (2478 rows).
2026-04-29 09:49:41,560 - INFO - Fetched data for FINCABLES (2478 rows).
2026-04-29 09:49:42,102 - INFO - Fetched data for GODFRYPHLP (2478 rows).
2026-04-29 09:49:42,675 - INFO - Fetched data for J&KBANK (2478 rows).
2026-04-29 09:49:43,296 - INFO - Fetched data for 

Final output path: Trials\Nifty_500_2025_Apr_20_stocks_results_gold_buy&hold_returns.xlsx


,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,NaN,3.750000,2023-04-01,3.750000,153.80,0.024382,12.0,75.000000
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,NaN,3.750000,2023-04-01,3.750000,405.15,0.009256,15.0,75.000000
2,2023-04-03,123.75,128.00,122.50,125.65,2365577,0,ANANTRAJ,NaN,3.750000,2023-04-01,3.750000,125.65,0.029845,19.0,75.000000
3,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,NaN,3.750000,2023-04-01,3.750000,169.20,0.022163,7.0,75.000000
4,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,NaN,3.750000,2023-04-01,3.750000,75.95,0.049375,3.0,75.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15976,2026-04-29,1094.00,1098.90,1088.45,1092.65,1289974,0,SBIN,0.001237,10.483329,2026-04-01,11.695538,1017.80,0.010676,12.0,209.394804
15977,2026-04-29,965.00,967.90,956.05,963.50,1248872,0,SHRIRAMFIN,0.010753,9.203623,2026-04-01,10.168204,900.55,0.010310,8.0,209.394804
15978,2026-04-29,1764.80,1767.60,1725.00,1725.90,86784,0,TORNTPOWER,-0.016525,8.008331,2026-04-01,10.584759,1337.10,0.005989,19.0,209.394804
15979,2026-04-29,170.46,172.23,169.63,170.52,3162671,0,UNIONBANK,0.000235,9.316553,2026-04-01,9.675143,171.64,0.060555,7.0,209.394804


In [6]:
old_df = final_df[~((final_df['Date']>'2025-11-30') & (final_df['Ticker']=='GOLDBEES'))]
old_df



,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,NaN,3.750000,2023-04-01,3.750000,153.80,0.024382,12.0,75.000000
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,NaN,3.750000,2023-04-01,3.750000,405.15,0.009256,15.0,75.000000
2,2023-04-03,123.75,128.00,122.50,125.65,2365577,0,ANANTRAJ,NaN,3.750000,2023-04-01,3.750000,125.65,0.029845,19.0,75.000000
3,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,NaN,3.750000,2023-04-01,3.750000,169.20,0.022163,7.0,75.000000
4,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,NaN,3.750000,2023-04-01,3.750000,75.95,0.049375,3.0,75.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15976,2026-04-29,1094.00,1098.90,1088.45,1092.65,1289974,0,SBIN,0.001237,10.483329,2026-04-01,11.695538,1017.80,0.010676,12.0,209.394804
15977,2026-04-29,965.00,967.90,956.05,963.50,1248872,0,SHRIRAMFIN,0.010753,9.203623,2026-04-01,10.168204,900.55,0.010310,8.0,209.394804
15978,2026-04-29,1764.80,1767.60,1725.00,1725.90,86784,0,TORNTPOWER,-0.016525,8.008331,2026-04-01,10.584759,1337.10,0.005989,19.0,209.394804
15979,2026-04-29,170.46,172.23,169.63,170.52,3162671,0,UNIONBANK,0.000235,9.316553,2026-04-01,9.675143,171.64,0.060555,7.0,209.394804


In [7]:
np.sort(old_df['Ticker'].unique())



array(['AAVAS', 'ABB', 'ABCAPITAL', 'ADANIPORTS', 'AIIL', 'AJANTPHARM',
       'ALIVUS', 'AMBER', 'ANANDRATHI', 'ANANTRAJ', 'ANGELONE',
       'APARINDS', 'APLAPOLLO', 'ASHOKLEY', 'ASTERDM', 'AUBANK',
       'AUROPHARMA', 'AXISBANK', 'BAJFINANCE', 'BANKBARODA', 'BANKINDIA',
       'BASF', 'BDL', 'BEL', 'BEML', 'BERGEPAINT', 'BHARATFORG',
       'BHARTIARTL', 'BHARTIHEXA', 'BHEL', 'BLUESTARCO', 'BPCL', 'BSE',
       'BSOFT', 'CANBK', 'CGPOWER', 'CHAMBLFERT', 'CHENNPETRO',
       'CHOLAFIN', 'CHOLAHLDNG', 'COALINDIA', 'COCHINSHIP', 'COFORGE',
       'COHANCE', 'COLPAL', 'CONCORDBIO', 'COROMANDEL', 'CUB',
       'CUMMINSIND', 'CYIENT', 'DATAPATTNS', 'DBREALTY', 'DEEPAKFERT',
       'DELHIVERY', 'DIVISLAB', 'DIXON', 'DLF', 'DOMS', 'ECLERX',
       'EICHERMOT', 'EIDPARRY', 'EIHOTEL', 'ELECON', 'EMAMILTD',
       'ENGINERSIN', 'ERIS', 'ETERNAL', 'EXIDEIND', 'FEDERALBNK',
       'FINCABLES', 'FLUOROCHEM', 'FORTIS', 'GESHIP', 'GLAND', 'GLENMARK',
       'GMDCLTD', 'GODFRYPHLP', 'GODREJPROP', '

In [8]:
# Build hedge book with the same monthly rebalancing rules used in the script
portfolio_end_date = pd.to_datetime(final_df['Date']).max()
df = build_rebalanced_hedge_book(
    base_portfolio_df=final_df,
    portfolio_end_date=portfolio_end_date,
    default_hedge_value=25.0
 )
df

(2026-04-29 09:52:16,284) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
(2026-04-29 09:52:16,284) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
(2026-04-29 09:52:16,284) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
2026-04-29 09:52:16,284 - WARNING - Connected successfully to TrueData Historical Data Service... 
(2026-04-29 09:52:35,668) ERROR :: <class 'requests.exceptions.ConnectionError'> -> ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)) (PID:7772 Thread:15436)
(2026-04-29 09:52:35,668) ERROR :: <class 'requests.exceptions.ConnectionError'> -> ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)) (PID:7772 Thread:15436)
(2026-04-29 09:52:35,668) ERROR :: <cla

,Date,Ticker,Open,Close,%change,Initial_Allocation,Buy_Hold_Value,Buy_Price,Quantity,Selection_Date,Hedge_Segment,Total_Portfolio_Value
0,2025-12-01,MOGSEC,62.70,62.82,NaN,10.297244,10.297244,62.82,0.163917,2025-12-01,2025-12_to_2026-01,20.594488
1,2025-12-01,SILVERBEES,166.02,166.21,NaN,10.297244,10.297244,166.21,0.061953,2025-12-01,2025-12_to_2026-01,20.594488
2,2025-12-02,MOGSEC,62.97,62.90,0.001273,10.297244,10.310357,62.82,0.163917,2025-12-01,2025-12_to_2026-01,20.553702
3,2025-12-02,SILVERBEES,167.01,165.34,-0.005234,10.297244,10.243345,166.21,0.061953,2025-12-01,2025-12_to_2026-01,20.553702
4,2025-12-03,MOGSEC,63.18,62.55,-0.005564,10.297244,10.252987,62.82,0.163917,2025-12-01,2025-12_to_2026-01,20.771404
...,...,...,...,...,...,...,...,...,...,...,...,...
138,2026-04-23,LIQUIDCASE,113.74,113.73,0.000088,16.640328,16.752275,112.97,0.147299,2026-03-01,2026-03_onward,16.752275
139,2026-04-24,LIQUIDCASE,113.79,113.78,0.000440,16.640328,16.759640,112.97,0.147299,2026-03-01,2026-03_onward,16.759640
140,2026-04-27,LIQUIDCASE,113.78,113.79,0.000088,16.640328,16.761113,112.97,0.147299,2026-03-01,2026-03_onward,16.761113
141,2026-04-28,LIQUIDCASE,113.82,113.80,0.000088,16.640328,16.762586,112.97,0.147299,2026-03-01,2026-03_onward,16.762586


In [9]:
# Hedge dataframe is already built in previous cell
df[['Date', 'Ticker', 'Buy_Hold_Value']].head()

,Date,Ticker,Buy_Hold_Value
0,2025-12-01,MOGSEC,10.297244
1,2025-12-01,SILVERBEES,10.297244
2,2025-12-02,MOGSEC,10.310357
3,2025-12-02,SILVERBEES,10.243345
4,2025-12-03,MOGSEC,10.252987


In [10]:
# No-op: valuation already computed in hedge builder cell
df.tail()

,Date,Ticker,Open,Close,%change,Initial_Allocation,Buy_Hold_Value,Buy_Price,Quantity,Selection_Date,Hedge_Segment,Total_Portfolio_Value
138,2026-04-23,LIQUIDCASE,113.74,113.73,0.000088,16.640328,16.752275,112.97,0.147299,2026-03-01,2026-03_onward,16.752275
139,2026-04-24,LIQUIDCASE,113.79,113.78,0.000440,16.640328,16.759640,112.97,0.147299,2026-03-01,2026-03_onward,16.759640
140,2026-04-27,LIQUIDCASE,113.78,113.79,0.000088,16.640328,16.761113,112.97,0.147299,2026-03-01,2026-03_onward,16.761113
141,2026-04-28,LIQUIDCASE,113.82,113.80,0.000088,16.640328,16.762586,112.97,0.147299,2026-03-01,2026-03_onward,16.762586
142,2026-04-29,LIQUIDCASE,113.82,113.81,0.000088,16.640328,16.764059,112.97,0.147299,2026-03-01,2026-03_onward,16.764059


In [11]:
conc_df = pd.concat([old_df, df])
conc_df



,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value,Hedge_Segment
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,NaN,3.750000,2023-04-01,3.750000,153.80,0.024382,12.0,75.000000,NaN
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,NaN,3.750000,2023-04-01,3.750000,405.15,0.009256,15.0,75.000000,NaN
2,2023-04-03,123.75,128.00,122.50,125.65,2365577.0,0.0,ANANTRAJ,NaN,3.750000,2023-04-01,3.750000,125.65,0.029845,19.0,75.000000,NaN
3,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,NaN,3.750000,2023-04-01,3.750000,169.20,0.022163,7.0,75.000000,NaN
4,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,NaN,3.750000,2023-04-01,3.750000,75.95,0.049375,3.0,75.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,2026-04-23,113.74,NaN,NaN,113.73,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.752275,112.97,0.147299,NaN,16.752275,2026-03_onward
139,2026-04-24,113.79,NaN,NaN,113.78,NaN,NaN,LIQUIDCASE,0.000440,16.640328,2026-03-01,16.759640,112.97,0.147299,NaN,16.759640,2026-03_onward
140,2026-04-27,113.78,NaN,NaN,113.79,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.761113,112.97,0.147299,NaN,16.761113,2026-03_onward
141,2026-04-28,113.82,NaN,NaN,113.80,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.762586,112.97,0.147299,NaN,16.762586,2026-03_onward


In [12]:
import sys
!{sys.executable} -m pip install plotly
import sys
!{sys.executable} -m pip install nbformat --upgrade
import plotly.express as px

# âœ… Group by Date and calculate total portfolio value
portfolio_summary = (
    conc_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)
# âœ… Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # ðŸ”‘ width
                  height=500)    # ðŸ”‘ height
import plotly.io as pio
pio.renderers.default = "browser"
fig.show()




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [13]:
# Momentum/Automating Momentum True Data/Trials/Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx



In [14]:
conc_df.to_excel(r'C:\Users\LENOVO\Desktop\Ocean Finvest\MOMENTUM_DB_2 copy\Trials\Nifty_500_2025_apr_20_stocks_results_goldsilverdebt_buyhold_returns.xlsx', index=True)
# \Trials



In [15]:
nse = fetch_truedata_history(
    ticker_list = ['BSE500'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
nse = nse[["Date", "Close"]].rename(columns={'Close':'Buy_Hold_Value'})
nse['%change'] = nse['Buy_Hold_Value'].pct_change()
nse = nse[nse['Date'] >= '2023-04-01']
nse.to_excel('Trials\\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx', index=False)
nse



(2026-04-29 09:52:57,879) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
(2026-04-29 09:52:57,879) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
(2026-04-29 09:52:57,879) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
(2026-04-29 09:52:57,879) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:7772 Thread:15436)
2026-04-29 09:52:57,879 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-04-29 09:52:58,341 - INFO - Fetched data for BSE500 (1241 rows).


,Date,Buy_Hold_Value,%change
480,2023-04-03,23226.81,0.002884
481,2023-04-05,23397.11,0.007332
482,2023-04-06,23475.05,0.003331
483,2023-04-10,23526.36,0.002186
484,2023-04-11,23652.45,0.005360
...,...,...,...
1236,2026-04-23,35729.71,-0.007772
1237,2026-04-24,35349.66,-0.010637
1238,2026-04-27,35751.09,0.011356
1239,2026-04-28,35650.27,-0.002820


In [16]:
conc_df




,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value,Hedge_Segment
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,NaN,3.750000,2023-04-01,3.750000,153.80,0.024382,12.0,75.000000,NaN
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,NaN,3.750000,2023-04-01,3.750000,405.15,0.009256,15.0,75.000000,NaN
2,2023-04-03,123.75,128.00,122.50,125.65,2365577.0,0.0,ANANTRAJ,NaN,3.750000,2023-04-01,3.750000,125.65,0.029845,19.0,75.000000,NaN
3,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,NaN,3.750000,2023-04-01,3.750000,169.20,0.022163,7.0,75.000000,NaN
4,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,NaN,3.750000,2023-04-01,3.750000,75.95,0.049375,3.0,75.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,2026-04-23,113.74,NaN,NaN,113.73,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.752275,112.97,0.147299,NaN,16.752275,2026-03_onward
139,2026-04-24,113.79,NaN,NaN,113.78,NaN,NaN,LIQUIDCASE,0.000440,16.640328,2026-03-01,16.759640,112.97,0.147299,NaN,16.759640,2026-03_onward
140,2026-04-27,113.78,NaN,NaN,113.79,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.761113,112.97,0.147299,NaN,16.761113,2026-03_onward
141,2026-04-28,113.82,NaN,NaN,113.80,NaN,NaN,LIQUIDCASE,0.000088,16.640328,2026-03-01,16.762586,112.97,0.147299,NaN,16.762586,2026-03_onward
